# Step 1 — Initialize Earth Engine 

In [22]:
import geemap
import ee

ee.Initialize(project="jaldrishti-508107")

print("Earth Engine initialized successfully!")


Earth Engine initialized successfully!


In [23]:
# Recreate the missing state boundary and geometry
maharashtra = (
    ee.FeatureCollection("FAO/GAUL/2025/level1")
    .filter(ee.Filter.eq("GAUL0_NAME", "India"))
    .filter(ee.Filter.eq("GAUL1_NAME", "Maharashtra"))
)

maharashtra_geometry = maharashtra.geometry()

# Continue with Sentinel-2 imagery for October 2025
sentinel2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(maharashtra_geometry)
    .filterDate("2025-10-01", "2025-11-01")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))
)

scene_count = sentinel2.size().getInfo()
print("Sentinel-2 scenes for Maharashtra:", scene_count)


Sentinel-2 scenes for Maharashtra: 271


# Step 2 — Create the Sentinel-2 composite

In [24]:

def mask_sentinel2_clouds(image):
    qa = image.select("QA60")

    cloud_bit = 1 << 10
    cirrus_bit = 1 << 11

    cloud_mask = (
        qa.bitwiseAnd(cloud_bit).eq(0)
        .And(qa.bitwiseAnd(cirrus_bit).eq(0))
    )

    # Keep the spectral bands needed for our analysis
    return (
        image.updateMask(cloud_mask)
        .select(["B2", "B3", "B4", "B8", "B11", "B12"])
        .divide(10000)
        .copyProperties(image, ["system:time_start"])
    )


# Apply cloud masking and build a monthly median composite
sentinel2_masked = sentinel2.map(mask_sentinel2_clouds)

sentinel2_composite = (
    sentinel2_masked
    .median()
    .clip(maharashtra_geometry)
)

print("Sentinel-2 composite created for Maharashtra.")


Sentinel-2 composite created for Maharashtra.


# Step 3 — Calculate the three indicators

In [25]:

# Vegetation index
ndvi = sentinel2_composite.normalizedDifference(
    ["B8", "B4"]
).rename("NDVI")

# Water-related spectral indicator
ndwi = sentinel2_composite.normalizedDifference(
    ["B3", "B8"]
).rename("NDWI")

# Modified water-related spectral indicator
mndwi = sentinel2_composite.normalizedDifference(
    ["B3", "B11"]
).rename("MNDWI")

print("NDVI, NDWI and MNDWI calculated.")


NDVI, NDWI and MNDWI calculated.


In [26]:
Map_ndvi = geemap.Map()
Map_ndvi.centerObject(maharashtra_geometry, 6)

Map_ndvi.addLayer(
    ndvi,
    {
        "min": -0.2,
        "max": 0.8,
        "palette": ["brown", "yellow", "green"]
    },
    "NDVI - Vegetation"
)

Map_ndvi.addLayer(
    maharashtra.style(
        color="FFFFFF",
        fillColor="00000000",
        width=2
    ),
    {},
    "Maharashtra Boundary"
)

Map_ndvi

Map(center=[19.452219941997686, 76.09119527148502], controls=(WidgetControl(options=['position', 'transparent_…

# calculate watershed-level NDVI

In [27]:
# Combine the three indices
watershed_indices = (
    ndvi.rename("NDVI")
    .addBands(ndwi.rename("NDWI"))
    .addBands(mndwi.rename("MNDWI"))
)

# Calculate mean indices for all clipped Maharashtra watersheds
indices_all = watershed_indices.reduceRegions(
    collection=basins_in_maharashtra,
    reducer=ee.Reducer.mean(),
    scale=100,
    tileScale=4
)

# Keep only the required attributes and index results
indices_export = indices_all.select([
    "HYBAS_ID",
    "NDVI",
    "NDWI",
    "MNDWI"
])

# Create the CSV export task
task = ee.batch.Export.table.toDrive(
    collection=indices_export,
    description="JalDrishti_Maharashtra_Sentinel2_Indices_Oct2025",
    folder="JalDrishti_AI",
    fileNamePrefix="maharashtra_watershed_indices_oct2025",
    fileFormat="CSV"
)

task.start()

print("Export task submitted.")
print("Task ID:", task.id)
print("Status:", task.status())

Export task submitted.
Task ID: RNX6VLGWLKPTOF7XD5FTBYKI
Status: {'state': 'READY', 'description': 'JalDrishti_Maharashtra_Sentinel2_Indices_Oct2025', 'priority': 100, 'creation_timestamp_ms': 1792091164600, 'update_timestamp_ms': 1792091164600, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'RNX6VLGWLKPTOF7XD5FTBYKI', 'name': 'projects/jaldrishti-508107/operations/RNX6VLGWLKPTOF7XD5FTBYKI'}


In [28]:
selected_fields = [
    "HYBAS_ID",
    "NEXT_DOWN",
    "MAIN_BAS",
    "SUB_AREA",
    "UP_AREA",
    "ele_mt_uav",
    "slp_dg_uav",
    "pre_mm_syr",
    "run_mm_syr"
]

basin_atlas = ee.FeatureCollection(
    "WWF/HydroATLAS/v1/Basins/level08"
)

watershed_boundaries = (
    basin_atlas
    .filterBounds(maharashtra_geometry)
    .select(selected_fields)
)

print("Watersheds restored:", watershed_boundaries.size().getInfo())

Watersheds restored: 522


In [30]:
# Restore the clipped watershed collection
basins_in_maharashtra = watershed_boundaries.map(
    lambda feature: feature.setGeometry(
        feature.geometry().intersection(
            maharashtra_geometry,
            ee.ErrorMargin(100)
        )
    )
)

# Test NDWI on only 5 watersheds at 100 m
ndwi_test = ndwi.reduceRegions(
    collection=basins_in_maharashtra.limit(5),
    reducer=ee.Reducer.mean(),
    scale=100,
    tileScale=4
)

print(ndwi_test.select(["HYBAS_ID", "mean"]).getInfo())

{'type': 'FeatureCollection', 'columns': {}, 'features': [{'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[76.68749949629996, 21.245832844078297], [76.68785181300653, 21.243407119305175], [76.69131202363704, 21.239924602467482], [76.6916688227371, 21.224999925628588], [76.6923153753791, 21.20081379925053], [76.70383771160344, 21.199137186617243], [76.70771704434753, 21.193826342708235], [76.70833242016583, 21.191668152493648], [76.70798016059955, 21.185077588162926], [76.70381533675945, 21.180881590603654], [76.70451987817107, 21.14757654691577], [76.71214493497727, 21.13992479251269], [76.7128540102, 21.135077746839507], [76.7204790031983, 21.127425872959204], [76.72118803150777, 21.10591075776759], [76.72464829658398, 21.102423773440496], [76.72535280958692, 21.055910834630026], [76.72952211762565, 21.051714825849192], [76.72881313627762, 21.039242726512914], [76.72118803150777, 21.031590892940933], [76.7204790031983, 21.02674389040042], [76.7128540102, 21.019092

In [ ]:
mndwi_test = mndwi.reduceRegions(
    collection=basins_in_maharashtra.limit(5),
    reducer=ee.Reducer.mean(),
    scale=100,
    tileScale=4
)

print(
    mndwi_test
    .select(["HYBAS_ID", "mean"])
    .getInfo()
)

In [ ]:
# Combine all three indices into one multi-band image
watershed_indices = (
    ndvi.rename("NDVI")
    .addBands(ndwi.rename("NDWI"))
    .addBands(mndwi.rename("MNDWI"))
)

# Test on 5 watersheds at 100 m resolution
indices_test = watershed_indices.reduceRegions(
    collection=basins_in_maharashtra.limit(5),
    reducer=ee.Reducer.mean(),
    scale=100,
    tileScale=4
)

# Display only the IDs and index values
print(
    indices_test
    .select(["HYBAS_ID", "NDVI", "NDWI", "MNDWI"])
    .getInfo()
)

In [33]:
# Test index calculations on 5 watersheds
test_basins = basins_in_maharashtra.limit(5)

test_results = (
    ndwi.rename("NDWI")
    .addBands(mndwi.rename("MNDWI"))
    .reduceRegions(
        collection=test_basins,
        reducer=ee.Reducer.mean(),
        scale=100,
        tileScale=4
    )
)

# Remove geometries so the output stays compact
test_table = test_results.map(
    lambda f: ee.Feature(
        None,
        f.toDictionary(["HYBAS_ID", "NDWI", "MNDWI"])
    )
)

print(test_table.getInfo())

{'type': 'FeatureCollection', 'columns': {}, 'features': [{'type': 'Feature', 'geometry': None, 'id': '00040000000000002ffb', 'properties': {'HYBAS_ID': 4081029590, 'MNDWI': -0.44442119254062595, 'NDWI': -0.5569030230774275}}, {'type': 'Feature', 'geometry': None, 'id': '00040000000000002b2a', 'properties': {'HYBAS_ID': 4081066200, 'MNDWI': -0.38701619258688374, 'NDWI': -0.47447245685719985}}, {'type': 'Feature', 'geometry': None, 'id': '00040000000000002b37', 'properties': {'HYBAS_ID': 4081026750, 'MNDWI': -0.40735246523538904, 'NDWI': -0.6210974800808103}}, {'type': 'Feature', 'geometry': None, 'id': '00040000000000002b38', 'properties': {'HYBAS_ID': 4081034280, 'MNDWI': -0.39194625640204045, 'NDWI': -0.5261447923959133}}, {'type': 'Feature', 'geometry': None, 'id': '00040000000000002b44', 'properties': {'HYBAS_ID': 4081024800, 'MNDWI': -0.41007348354104, 'NDWI': -0.6051945092456685}}]}


In [34]:
# Check the existing three-index export task
print(task.status())

{'state': 'COMPLETED', 'description': 'JalDrishti_Maharashtra_Sentinel2_Indices_Oct2025', 'priority': 100, 'creation_timestamp_ms': 1792091164600, 'update_timestamp_ms': 1792091601549, 'start_timestamp_ms': 1792091183410, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['https://drive.google.com/#folders/1kL9lMyylc9C8sFbOBTk6snGk0lhYtxUv'], 'attempt': 1, 'batch_eecu_usage_seconds': 19151.603515625, 'id': 'RNX6VLGWLKPTOF7XD5FTBYKI', 'name': 'projects/jaldrishti-508107/operations/RNX6VLGWLKPTOF7XD5FTBYKI'}


In [35]:
import pandas as pd
import geemap
import ee

# Approximate study areas (longitude, latitude)
areas = ee.FeatureCollection([
    ee.Feature(
        ee.Geometry.Point([72.8402, 19.0544]).buffer(2000),
        {"Area": "Bandra, Mumbai", "Buffer_km": 2}
    ),
    ee.Feature(
        ee.Geometry.Point([73.6750, 18.7350]).buffer(3000),
        {"Area": "Talegaon Dabhade", "Buffer_km": 3}
    )
])

# Calculate mean indices for each named study area
area_results = watershed_indices.reduceRegions(
    collection=areas,
    reducer=ee.Reducer.mean(),
    scale=100,
    tileScale=4
)

# Retrieve only table properties, not geometries
area_table = area_results.map(
    lambda f: ee.Feature(
        None,
        f.toDictionary([
            "Area", "Buffer_km", "NDVI", "NDWI", "MNDWI"
        ])
    )
)

result = area_table.getInfo()

rows = [
    feature["properties"]
    for feature in result["features"]
]

df_areas = pd.DataFrame(rows)
display(df_areas.round(4))

# Display the areas on a map
Map = geemap.Map()
Map.centerObject(areas, 9)
Map.addLayer(
    ndvi.clip(areas.geometry()),
    {"min": 0, "max": 0.8,
     "palette": ["brown", "yellow", "green"]},
    "NDVI — October 2025"
)
Map.addLayer(
    areas.style(
        color="red",
        fillColor="00000000",
        width=2
    ),
    {},
    "Named study areas"
)
Map

,Area,Buffer_km,MNDWI,NDVI,NDWI
0,"Bandra, Mumbai",2,-0.2949,0.3163,-0.3219
1,Talegaon Dabhade,3,-0.4008,0.4693,-0.4518


Map(center=[18.833675075984797, 73.41848013371161], controls=(WidgetControl(options=['position', 'transparent_…

In [37]:
%pip install geopandas

   ---------------------------------------- 0.0/23.8 MB ? eta -:--:--
    --------------------------------------- 0.5/23.8 MB 2.5 MB/s eta 0:00:10
   - -------------------------------------- 0.8/23.8 MB 2.8 MB/s eta 0:00:09
   -- ------------------------------------- 1.3/23.8 MB 2.2 MB/s eta 0:00:11
   -- ------------------------------------- 1.6/23.8 MB 1.9 MB/s eta 0:00:12
   --- ------------------------------------ 2.1/23.8 MB 2.3 MB/s eta 0:00:10
   --- ------------------------------------ 2.4/23.8 MB 2.1 MB/s eta 0:00:11
   --- ------------------------------------ 2.4/23.8 MB 2.1 MB/s eta 0:00:11
   --- ------------------------------------ 2.4/23.8 MB 2.1 MB/s eta 0:00:11
   --- ------------------------------------ 2.4/23.8 MB 2.1 MB/s eta 0:00:11
   ---- ----------------------------------- 2.6/23.8 MB 1.3 MB/s eta 0:00:16
   ---- ----------------------------------- 2.6/23.8 MB 1.3 MB/s eta 0:00:16
   ---- ----------------------------------- 2.6/23.8 MB 1.3 MB/s eta 0:00:16
   ---

In [38]:
import geopandas as gpd
print("GeoPandas version:", gpd.__version__)

GeoPandas version: 1.2.0


In [39]:
import geopandas as gpd

# Change this to the folder containing the four shapefile components
shapefile_path = r"C:\path\to\your\folder\vb_soi_mh.shp"

# Load the village boundaries
villages = gpd.read_file(shapefile_path)

# Standardize the coordinate system for mapping and satellite analysis
villages = villages.to_crs("EPSG:4326")

# Display the key administrative fields
print("Total boundary records:", len(villages))
print("Coordinate system:", villages.crs)
print("Available columns:", villages.columns.tolist())

display(
    villages[
        ["village", "district", "subdistric", "block",
         "total_urba", "geometry"]
    ].head(10)
)

DataSourceError: C:\path\to\your\folder\vb_soi_mh.shp: No such file or directory

In [40]:
from pathlib import Path

# Search common locations for the uploaded shapefile
search_dirs = [
    Path.cwd(),
    Path.home() / "Downloads",
    Path.home() / "OneDrive" / "Desktop",
    Path.home() / "Desktop",
]

matches = []

for folder in search_dirs:
    if folder.exists():
        matches.extend(folder.rglob("vb_soi_mh.shp"))

print("Shapefile matches:")
for path in matches:
    print(path)

Shapefile matches:
C:\Users\MITALI SALHOTRA\Downloads\vb_soi_mh.shp


In [41]:
import geopandas as gpd

shapefile_path = r"C:\Users\MITALI SALHOTRA\Downloads\vb_soi_mh.shp"

villages = gpd.read_file(shapefile_path)
villages = villages.to_crs("EPSG:4326")

print("Total boundary records:", len(villages))
print("Available columns:", villages.columns.tolist())

display(villages.drop(columns="geometry").head(10))

Total boundary records: 44468
Available columns: ['id', 'objectid_1', 'objectid', 'village', 'vlcode', 'gram_panch', 'gram_pan_1', 'block', 'bkcode', 'subdistric', 'sdcode', 'sub_distri', 'district', 'dtcode', 'district_h', 'state', 'stcode', 'uqcode', 'country', 'total_urba', 'total_hous', 'total_popu', 'avg_househ', 'total_geog', 'total_male', 'total_fema', 'tapwater_t', 'tapwater_u', 'covered_we', 'covered__1', 'covered__2', 'uncovered_', 'uncovere_1', 'uncovere_2', 'handpump_s', 'hand_pump_', 'hand_pum_1', 'tubewell_b', 'tube_wells', 'tube_wel_1', 'spring_sta', 'spring_fun', 'spring_f_1', 'river_cana', 'tank_pond_', 'others_sta', 'closed_dra', 'open_drain', 'village_pi', 'forest_are', 'area_under', 'barren_unc', 'permanent_', 'land_under', 'culturable', 'fallows_la', 'current_fa', 'net_area_s', 'total_unir', 'area_irrig', 'canals_are', 'wells_tube', 'tanks_lake', 'waterfall_', 'other_sour', 'nearest_to', 'nearest__1', 'shape_leng', 'shape_le_1', 'shape_area', 'ds_name', 'src_agency

,id,objectid_1,objectid,village,vlcode,gram_panch,gram_pan_1,block,bkcode,subdistric,...,waterfall_,other_sour,nearest_to,nearest__1,shape_leng,shape_le_1,shape_area,ds_name,src_agency,state_name
0,1209,1209.0,64052.0,Kurdhe,565655,None,None,Ratnagiri,322,Ratnagiri,...,0.0,0.0,Ratnagiri,20.0,11048.441143,11048.441143,3.577363e+06,Administrative Boundary,Survey of India (SOI),Maharashtra\n
1,1210,1210.0,64053.0,Shivar Ambere,565664,None,None,Ratnagiri,322,Ratnagiri,...,0.0,0.0,Ratnagiri,36.0,16279.927582,16279.927582,7.667603e+06,Administrative Boundary,Survey of India (SOI),Maharashtra\n
2,1211,1211.0,64054.0,Thikanbehere,565656,None,None,Ratnagiri,322,Ratnagiri,...,0.0,0.0,Ratnagiri,18.0,4041.095339,4041.095339,1.013724e+06,Administrative Boundary,Survey of India (SOI),Maharashtra\n
3,1212,1212.0,64055.0,Mervi,565660,None,None,Ratnagiri,322,Ratnagiri,...,0.0,0.0,Ratnagiri,20.0,12132.375524,12132.375524,5.457113e+06,Administrative Boundary,Survey of India (SOI),Maharashtra\n
4,1213,1213.0,64056.0,Lavgan,565948,None,None,Lanja,324,Lanja,...,0.0,0.0,Ratnagiri,30.0,9688.207746,9688.207746,3.700227e+06,Administrative Boundary,Survey of India (SOI),Maharashtra\n
5,1215,1215.0,64058.0,Jambhul Aad,565659,None,None,Ratnagiri,322,Ratnagiri,...,0.0,0.0,Ratnagiri,21.0,4395.042469,4395.042469,8.817642e+05,Administrative Boundary,Survey of India (SOI),Maharashtra\n
6,1216,1216.0,64059.0,Gavade Ambere,565663,None,None,Ratnagiri,322,Ratnagiri,...,0.0,0.0,Ratnagiri,30.0,14292.104144,14292.104144,7.011139e+06,Administrative Boundary,Survey of India (SOI),Maharashtra\n
7,1217,1217.0,64060.0,Peth Purnagad,565662,None,None,Ratnagiri,322,Ratnagiri,...,0.0,0.0,Ratnagiri,23.0,2827.792080,2827.792080,3.840004e+05,Administrative Boundary,Survey of India (SOI),Maharashtra\n
8,1218,1218.0,64061.0,Dabhil Ambere,565668,None,None,Ratnagiri,322,Ratnagiri,...,0.0,0.0,Ratnagiri,42.0,16015.026283,16015.026283,1.282225e+07,Administrative Boundary,Survey of India (SOI),Maharashtra\n
9,1219,1219.0,64062.0,Muslimwadi,565665,None,None,Ratnagiri,322,Ratnagiri,...,0.0,0.0,Ratnagiri,25.0,4153.340140,4153.340140,8.765737e+05,Administrative Boundary,Survey of India (SOI),Maharashtra\n


In [42]:
# Check state labels and missing village names
print("State names:")
print(villages["state_name"].value_counts(dropna=False).head(10))

print("\nMissing village names:", villages["village"].isna().sum())
print("Unique village names:", villages["village"].nunique())
print("Unique districts:", villages["district"].nunique())
print("Unique subdistricts:", villages["subdistric"].nunique())

# Preview the administrative hierarchy
display(
    villages[
        ["village", "district", "subdistric", "block", "vlcode"]
    ].head(15)
)

State names:
state_name
Maharashtra\n    44468
Name: count, dtype: int64

Missing village names: 0
Unique village names: 29050
Unique districts: 36
Unique subdistricts: 350


,village,district,subdistric,block,vlcode
0,Kurdhe,Ratnagiri,Ratnagiri,Ratnagiri,565655
1,Shivar Ambere,Ratnagiri,Ratnagiri,Ratnagiri,565664
2,Thikanbehere,Ratnagiri,Ratnagiri,Ratnagiri,565656
3,Mervi,Ratnagiri,Ratnagiri,Ratnagiri,565660
4,Lavgan,Ratnagiri,Lanja,Lanja,565948
5,Jambhul Aad,Ratnagiri,Ratnagiri,Ratnagiri,565659
6,Gavade Ambere,Ratnagiri,Ratnagiri,Ratnagiri,565663
7,Peth Purnagad,Ratnagiri,Ratnagiri,Ratnagiri,565662
8,Dabhil Ambere,Ratnagiri,Ratnagiri,Ratnagiri,565668
9,Muslimwadi,Ratnagiri,Ratnagiri,Ratnagiri,565665


In [43]:
# Keep a copy of the original boundary data
villages_raw = villages.copy()

# Remove records without geometry or with empty geometry
villages_clean = villages[
    villages.geometry.notna()
    & ~villages.geometry.is_empty
].copy()

# Normalize administrative text fields
for col in ["village", "district", "subdistric", "block"]:
    villages_clean[col] = villages_clean[col].astype("string").str.strip()

# Check village-code duplicates
duplicate_codes = villages_clean[
    villages_clean["vlcode"].duplicated(keep=False)
]

print("Original records:", len(villages_raw))
print("Records with usable geometry:", len(villages_clean))
print("Records removed:", len(villages_raw) - len(villages_clean))
print("Duplicate village-code records:", len(duplicate_codes))
print("Unique districts:", villages_clean["district"].nunique())
print("Unique subdistricts:", villages_clean["subdistric"].nunique())

display(
    villages_clean[
        ["village", "district", "subdistric", "vlcode", "geometry"]
    ].head(5)
)

Original records: 44468
Records with usable geometry: 44468
Records removed: 0
Duplicate village-code records: 267
Unique districts: 36
Unique subdistricts: 350


,village,district,subdistric,vlcode,geometry
0,Kurdhe,Ratnagiri,Ratnagiri,565655,"POLYGON ((73.29159 16.85249, 73.29516 16.85232..."
1,Shivar Ambere,Ratnagiri,Ratnagiri,565664,"POLYGON ((73.39141 16.8392, 73.38743 16.83798,..."
2,Thikanbehere,Ratnagiri,Ratnagiri,565656,"POLYGON ((73.32236 16.84727, 73.32174 16.84361..."
3,Mervi,Ratnagiri,Ratnagiri,565660,"POLYGON ((73.32059 16.83938, 73.32062 16.83872..."
4,Lavgan,Ratnagiri,Lanja,565948,"POLYGON ((73.39225 16.82555, 73.39256 16.82011..."


In [44]:
# Inspect records with duplicate village codes
duplicate_codes = villages_clean[
    villages_clean["vlcode"].duplicated(keep=False)
].sort_values(["vlcode", "district", "subdistric"])

display(
    duplicate_codes[
        ["vlcode", "village", "district", "subdistric", "block"]
    ].head(30)
)

print("Duplicate records:", len(duplicate_codes))
print("Distinct duplicated codes:", duplicate_codes["vlcode"].nunique())


,vlcode,village,district,subdistric,block
6682,251488,Greater Mumbai,Mumbai,Mumbai,<NA>
42687,251488,Greater Mumbai,Mumbai Suburban,Mumbai Suburban,<NA>
10204,999999,Water Body,Ahmednagar,Akola,<NA>
13188,999999,Water Body,Ahmednagar,Akola,<NA>
31667,999999,Reserved Forest,Amravati,Chikhaldara,<NA>
31699,999999,Reserved Forest,Amravati,Chikhaldara,<NA>
31700,999999,Reserved Forest,Amravati,Chikhaldara,<NA>
31320,999999,Reserved Forest,Amravati,Dharni,<NA>
31695,999999,Reserved Forest,Amravati,Dharni,<NA>
32017,999999,Reserved Forest,Amravati,Dharni,<NA>


Duplicate records: 267
Distinct duplicated codes: 2


In [45]:
# Preserve every boundary record
villages_clean = villages_clean.copy()

# Create a unique row ID independent of the source village code
villages_clean = villages_clean.reset_index(drop=True)
villages_clean["area_id"] = [
    f"MH_{i:05d}" for i in range(1, len(villages_clean) + 1)
]

# Flag source codes that cannot uniquely identify a village
villages_clean["code_status"] = "source_code"

villages_clean.loc[
    villages_clean["vlcode"].astype("string") == "999999",
    "code_status"
] = "unassigned_source_code"

villages_clean.loc[
    villages_clean["village"].str.strip().str.lower().isin(
        ["greater mumbai", "water body", "reserved forest"]
    ),
    "code_status"
] = "special_area_review"

print("Total preserved polygons:", len(villages_clean))
print("Unique area IDs:", villages_clean["area_id"].nunique())
print("\nRecord categories:")
print(villages_clean["code_status"].value_counts())

display(
    villages_clean[
        ["area_id", "village", "district", "subdistric",
         "vlcode", "code_status"]
    ].head(10)
)

Total preserved polygons: 44468
Unique area IDs: 44468

Record categories:
code_status
source_code               44201
special_area_review         167
unassigned_source_code      100
Name: count, dtype: int64


,area_id,village,district,subdistric,vlcode,code_status
0,MH_00001,Kurdhe,Ratnagiri,Ratnagiri,565655,source_code
1,MH_00002,Shivar Ambere,Ratnagiri,Ratnagiri,565664,source_code
2,MH_00003,Thikanbehere,Ratnagiri,Ratnagiri,565656,source_code
3,MH_00004,Mervi,Ratnagiri,Ratnagiri,565660,source_code
4,MH_00005,Lavgan,Ratnagiri,Lanja,565948,source_code
5,MH_00006,Jambhul Aad,Ratnagiri,Ratnagiri,565659,source_code
6,MH_00007,Gavade Ambere,Ratnagiri,Ratnagiri,565663,source_code
7,MH_00008,Peth Purnagad,Ratnagiri,Ratnagiri,565662,source_code
8,MH_00009,Dabhil Ambere,Ratnagiri,Ratnagiri,565668,source_code
9,MH_00010,Muslimwadi,Ratnagiri,Ratnagiri,565665,source_code


In [46]:

# Check geometry validity before spatial analysis
print("CRS:", villages_clean.crs)
print("Missing geometries:", villages_clean.geometry.isna().sum())
print("Empty geometries:", villages_clean.geometry.is_empty.sum())
print("Invalid geometries:", (~villages_clean.geometry.is_valid).sum())
print("Zero-area geometries:", (villages_clean.geometry.area == 0).sum())

print("\nGeometry types:")
print(villages_clean.geometry.geom_type.value_counts())


CRS: EPSG:4326
Missing geometries: 0
Empty geometries: 0
Invalid geometries: 5
Zero-area geometries: 0

Geometry types:
Polygon         44406
MultiPolygon       62
Name: count, dtype: int64


In [47]:
# Repair invalid geometries without dropping any boundary records
villages_clean = villages_clean.copy()

invalid_before = ~villages_clean.geometry.is_valid
print("Invalid geometries before repair:", invalid_before.sum())

villages_clean.loc[invalid_before, "geometry"] = (
    villages_clean.loc[invalid_before, "geometry"].make_valid()
)

print("Total polygons after repair:", len(villages_clean))
print("Unique area IDs:", villages_clean["area_id"].nunique())
print("Invalid geometries remaining:", (~villages_clean.geometry.is_valid).sum())
print("Empty geometries after repair:", villages_clean.geometry.is_empty.sum())
print("\nGeometry types after repair:")
print(villages_clean.geometry.geom_type.value_counts())


Invalid geometries before repair: 5
Total polygons after repair: 44468
Unique area IDs: 44468
Invalid geometries remaining: 0
Empty geometries after repair: 0

Geometry types after repair:
Polygon         44406
MultiPolygon       62
Name: count, dtype: int64


In [48]:

from pathlib import Path

output_path = Path.home() / "Downloads" / "maharashtra_village_boundaries_clean.gpkg"

villages_clean.to_file(
    output_path,
    driver="GPKG",
    layer="maharashtra_villages"
)

print("Saved to:", output_path)
print("Records saved:", len(villages_clean))
print("Unique area IDs:", villages_clean["area_id"].nunique())
print("File size (MB):", round(output_path.stat().st_size / (1024 * 1024), 2))


Saved to: C:\Users\MITALI SALHOTRA\Downloads\maharashtra_village_boundaries_clean.gpkg
Records saved: 44468
Unique area IDs: 44468
File size (MB): 78.52


In [2]:

from pathlib import Path
import geopandas as gpd
import zipfile

# Reload the already-cleaned boundary file
gpkg_path = Path.home() / "Downloads" / "maharashtra_village_boundaries_clean.gpkg"

villages_clean = gpd.read_file(
    gpkg_path,
    layer="maharashtra_villages"
)

print("Records loaded:", len(villages_clean))
print("Unique area IDs:", villages_clean["area_id"].nunique())

# Prepare the Earth Engine upload files
output_dir = Path.home() / "Downloads" / "jaldrishti_ee_upload"
output_dir.mkdir(parents=True, exist_ok=True)

shp_path = output_dir / "maharashtra_villages.shp"
zip_path = Path.home() / "Downloads" / "maharashtra_villages_ee.zip"

villages_clean.to_file(
    shp_path,
    driver="ESRI Shapefile",
    encoding="UTF-8"
)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for ext in [".shp", ".shx", ".dbf", ".prj"]:
        component = shp_path.with_suffix(ext)
        if not component.exists():
            raise FileNotFoundError(f"Missing component: {component}")
        zf.write(component, arcname=component.name)

print("ZIP created:", zip_path)
print("ZIP size (MB):", round(zip_path.stat().st_size / (1024 * 1024), 2))


Records loaded: 44468
Unique area IDs: 44468


C:\Users\MITALI SALHOTRA\AppData\Local\Temp\ipykernel_14832\1883429770.py:23: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  villages_clean.to_file(
C:\Users\MITALI SALHOTRA\anaconda3\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'code_status' to 'code_statu'
  ogr_write(


ZIP created: C:\Users\MITALI SALHOTRA\Downloads\maharashtra_villages_ee.zip
ZIP size (MB): 29.88


In [3]:

import ee

ee.Initialize(project="jaldrishti-508107")

village_asset = ee.FeatureCollection(
    "projects/jaldrishti-508107/assets/maharashtra_villages"
)

print("Uploaded boundary count:", village_asset.size().getInfo())
print(
    "Sample records:",
    village_asset.limit(3)
    .select(["area_id", "village", "district", "vlcode"])
    .getInfo()
)


Uploaded boundary count: 44468
Sample records: {'type': 'FeatureCollection', 'columns': {}, 'version': 1792257505968850.0, 'id': 'projects/jaldrishti-508107/assets/maharashtra_villages', 'properties': {'system:asset_size': 27530551}, 'features': [{'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[76.56630558887885, 18.440420656671865], [76.56830772874216, 18.4383114924636], [76.56809820393944, 18.437660518985982], [76.566831788544, 18.433972778194224], [76.5653201161366, 18.43155599387735], [76.56775931272063, 18.43121258973803], [76.57779674510746, 18.431912738262408], [76.58209972265782, 18.43232741413651], [76.5930155875075, 18.430365385782277], [76.60562598921608, 18.42994179694494], [76.60763253965263, 18.429041030825726], [76.60993349699778, 18.427342140456265], [76.60705731588067, 18.43237202440168], [76.60573742098786, 18.434927056443268], [76.60367732803475, 18.43508759772143], [76.60030183523305, 18.436193469373897], [76.5946521287671, 18.440121918441797], 

In [4]:

import ee

ee.Initialize(project="jaldrishti-508107")

# Maharashtra boundary
maharashtra = (
    ee.FeatureCollection("FAO/GAUL/2025/level1")
    .filter(ee.Filter.eq("GAUL0_NAME", "India"))
    .filter(ee.Filter.eq("GAUL1_NAME", "Maharashtra"))
)

maharashtra_geometry = maharashtra.geometry()

# October 2025 Sentinel-2 surface reflectance
sentinel2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(maharashtra_geometry)
    .filterDate("2025-10-01", "2025-11-01")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))
)

print("Sentinel-2 scenes:", sentinel2.size().getInfo())

# Build the composite
sentinel2_composite = sentinel2.median().clip(maharashtra_geometry)

# Calculate indices
ndvi = sentinel2_composite.normalizedDifference(
    ["B8", "B4"]
).rename("NDVI")

ndwi = sentinel2_composite.normalizedDifference(
    ["B3", "B8"]
).rename("NDWI")

mndwi = sentinel2_composite.normalizedDifference(
    ["B3", "B11"]
).rename("MNDWI")

watershed_indices = ndvi.addBands(ndwi).addBands(mndwi)

print("Bands ready:", watershed_indices.bandNames().getInfo())


Sentinel-2 scenes: 271
Bands ready: ['NDVI', 'NDWI', 'MNDWI']


In [5]:

# Mask clouds, cirrus, cloud shadows, snow and invalid SCL pixels
def mask_sentinel2_scl(image):
    scl = image.select("SCL")

    clear_pixels = (
        scl.neq(0)     # No data
        .And(scl.neq(1))   # Saturated / defective
        .And(scl.neq(3))   # Cloud shadows
        .And(scl.neq(7))   # Low-probability cloud / unclassified
        .And(scl.neq(8))   # Medium-probability clouds
        .And(scl.neq(9))   # High-probability clouds
        .And(scl.neq(10))  # Cirrus
        .And(scl.neq(11))  # Snow / ice
    )

    return image.updateMask(clear_pixels)

# Rebuild the October composite with pixel-level masking
sentinel2_masked = sentinel2.map(mask_sentinel2_scl)

sentinel2_composite = (
    sentinel2_masked
    .median()
    .clip(maharashtra_geometry)
)

# Recalculate all three indices from the masked composite
ndvi = sentinel2_composite.normalizedDifference(
    ["B8", "B4"]
).rename("NDVI")

ndwi = sentinel2_composite.normalizedDifference(
    ["B3", "B8"]
).rename("NDWI")

mndwi = sentinel2_composite.normalizedDifference(
    ["B3", "B11"]
).rename("MNDWI")

village_indices_image = ndvi.addBands(ndwi).addBands(mndwi)

print("Scenes used:", sentinel2_masked.size().getInfo())
print("Prepared bands:", village_indices_image.bandNames().getInfo())


Scenes used: 271
Prepared bands: ['NDVI', 'NDWI', 'MNDWI']


In [6]:

# Load the uploaded village boundaries
village_asset = ee.FeatureCollection(
    "projects/jaldrishti-508107/assets/maharashtra_villages"
)

# Calculate mean NDVI, NDWI and MNDWI per polygon
village_stats = village_indices_image.reduceRegions(
    collection=village_asset,
    reducer=ee.Reducer.mean(),
    scale=100,
    tileScale=4
)

# Export only the required identifiers and index values
village_export = village_stats.map(
    lambda feature: ee.Feature(
        None,
        feature.toDictionary([
            "area_id",
            "village",
            "district",
            "subdistric",
            "vlcode",
            "NDVI",
            "NDWI",
            "MNDWI"
        ])
    )
)

task = ee.batch.Export.table.toDrive(
    collection=village_export,
    description="JalDrishti_Maharashtra_Village_Indices_Oct2025",
    folder="JalDrishti_AI",
    fileNamePrefix="maharashtra_village_indices_oct2025",
    fileFormat="CSV",
    selectors=[
        "area_id", "village", "district", "subdistric",
        "vlcode", "NDVI", "NDWI", "MNDWI"
    ]
)

task.start()
print("Export task started.")
print("Task ID:", task.id)
print("Expected boundary records:", village_asset.size().getInfo())


Export task started.
Task ID: KP5KD56Y2HUFJB6ZMD3LCFGJ
Expected boundary records: 44468


In [7]:

task_status = ee.data.getTaskStatus("KP5KD56Y2HUFJB6ZMD3LCFGJ")
print(task_status)


[{'state': 'RUNNING', 'description': 'JalDrishti_Maharashtra_Village_Indices_Oct2025', 'priority': 100, 'creation_timestamp_ms': 1792258026165, 'update_timestamp_ms': 1792258049674, 'start_timestamp_ms': 1792258029425, 'task_type': 'EXPORT_FEATURES', 'attempt': 1, 'batch_eecu_usage_seconds': 11.114731788, 'id': 'KP5KD56Y2HUFJB6ZMD3LCFGJ', 'name': 'projects/jaldrishti-508107/operations/KP5KD56Y2HUFJB6ZMD3LCFGJ'}]


In [8]:
print(ee.data.getTaskStatus("KP5KD56Y2HUFJB6ZMD3LCFGJ"))

[{'state': 'RUNNING', 'description': 'JalDrishti_Maharashtra_Village_Indices_Oct2025', 'priority': 100, 'creation_timestamp_ms': 1792258026165, 'update_timestamp_ms': 1792258085700, 'start_timestamp_ms': 1792258029425, 'task_type': 'EXPORT_FEATURES', 'attempt': 1, 'id': 'KP5KD56Y2HUFJB6ZMD3LCFGJ', 'name': 'projects/jaldrishti-508107/operations/KP5KD56Y2HUFJB6ZMD3LCFGJ'}]


In [9]:
status = ee.data.getTaskStatus("KP5KD56Y2HUFJB6ZMD3LCFGJ")
print(status)

[{'state': 'COMPLETED', 'description': 'JalDrishti_Maharashtra_Village_Indices_Oct2025', 'priority': 100, 'creation_timestamp_ms': 1792258026165, 'update_timestamp_ms': 1792258086129, 'start_timestamp_ms': 1792258029425, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['https://drive.google.com/#folders/1kL9lMyylc9C8sFbOBTk6snGk0lhYtxUv'], 'attempt': 1, 'batch_eecu_usage_seconds': 2582.358642578125, 'id': 'KP5KD56Y2HUFJB6ZMD3LCFGJ', 'name': 'projects/jaldrishti-508107/operations/KP5KD56Y2HUFJB6ZMD3LCFGJ'}]


In [11]:

import pandas as pd
from pathlib import Path

csv_path = (
    Path.home()
    / "Downloads"
    / "maharashtra_village_indices_oct2025.csv"
)

village_df = pd.read_csv(csv_path)

index_columns = ["NDVI", "NDWI", "MNDWI"]

# Find records where all three indices are missing
missing_mask = village_df[index_columns].isna().all(axis=1)
missing_areas = village_df.loc[missing_mask].copy()

print("Total exported records:", len(village_df))
print("Areas missing all three indices:", len(missing_areas))
print("\nMissing areas by district:")
print(missing_areas["district"].value_counts().head(10))

print("\nSample missing areas:")
display(
    missing_areas[
        ["area_id", "village", "district", "subdistric", "vlcode"]
    ].head(10)
)


Total exported records: 44468
Areas missing all three indices: 1109

Missing areas by district:
district
Kolhapur      492
Sindhudurg    435
Gadchiroli    119
Ratnagiri      28
Sangli         13
Chandrapur     10
Amravati        8
Wardha          1
Solapur         1
Palghar         1
Name: count, dtype: int64

Sample missing areas:


,area_id,village,district,subdistric,vlcode
141,MH_43764,Majgaon,Sindhudurg,Sawantwadi,566922
144,MH_44287,Banda,Sindhudurg,Sawantwadi,566923
145,MH_44289,Ajra,Kolhapur,Ajra,567964
471,MH_42974,Teradoh,Gadchiroli,Etapalli,539679
493,MH_38451,Garewada,Gadchiroli,Bhamragad,539788
503,MH_36260,Bondra M,Gadchiroli,Aheri,539994
526,MH_36161,Bhogapur W L,Gadchiroli,Sironcha,540063
527,MH_36185,Bramhanpalli Ry,Gadchiroli,Sironcha,540073
528,MH_37148,Sirkonda Ch No 2,Gadchiroli,Sironcha,540038
534,MH_37154,Rangayapalli W B,Gadchiroli,Sironcha,540051


In [12]:

import pandas as pd
import ee
from pathlib import Path

ee.Initialize(project="jaldrishti-508107")

# Load the CSV and identify areas missing all three indices
csv_path = Path.home() / "Downloads" / "maharashtra_village_indices_oct2025.csv"
village_df = pd.read_csv(csv_path)

index_columns = ["NDVI", "NDWI", "MNDWI"]
missing_mask = village_df[index_columns].isna().all(axis=1)
missing_ids = village_df.loc[missing_mask, "area_id"].dropna().tolist()

print("Areas to diagnose:", len(missing_ids))

# Select only the affected polygons from the uploaded asset
village_asset = ee.FeatureCollection(
    "projects/jaldrishti-508107/assets/maharashtra_villages"
)

missing_features = village_asset.filter(
    ee.Filter.inList("area_id", missing_ids)
)

# Count valid pixels in the masked NDVI image at 100 m scale
valid_pixel_counts = ndvi.rename("valid_pixel_count").reduceRegions(
    collection=missing_features,
    reducer=ee.Reducer.count(),
    scale=100,
    tileScale=4
)

# Keep identifiers and the pixel count
diagnostic_table = valid_pixel_counts.map(
    lambda feature: ee.Feature(
        None,
        feature.toDictionary([
            "area_id", "village", "district",
            "subdistric", "vlcode", "valid_pixel_count"
        ])
    )
)

# Export a separate diagnostic CSV
diagnostic_task = ee.batch.Export.table.toDrive(
    collection=diagnostic_table,
    description="JalDrishti_Missing_Area_Pixel_Diagnostic_Oct2025",
    folder="JalDrishti_AI",
    fileNamePrefix="missing_area_pixel_diagnostic_oct2025",
    fileFormat="CSV",
    selectors=[
        "area_id", "village", "district",
        "subdistric", "vlcode", "valid_pixel_count"
    ]
)

diagnostic_task.start()
print("Diagnostic export started.")
print("Task ID:", diagnostic_task.id)


Areas to diagnose: 1109
Diagnostic export started.
Task ID: LTTPTD5K5RIWF6GTYRLD3PC7


In [13]:

print(ee.data.getTaskStatus("LTTPTD5K5RIWF6GTYRLD3PC7"))


[{'state': 'COMPLETED', 'description': 'JalDrishti_Missing_Area_Pixel_Diagnostic_Oct2025', 'priority': 100, 'creation_timestamp_ms': 1792258465993, 'update_timestamp_ms': 1792258479606, 'start_timestamp_ms': 1792258469169, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['https://drive.google.com/#folders/1kL9lMyylc9C8sFbOBTk6snGk0lhYtxUv'], 'attempt': 1, 'batch_eecu_usage_seconds': 102.57054901123047, 'id': 'LTTPTD5K5RIWF6GTYRLD3PC7', 'name': 'projects/jaldrishti-508107/operations/LTTPTD5K5RIWF6GTYRLD3PC7'}]


In [14]:

import ee
import pandas as pd
from pathlib import Path

ee.Initialize(project="jaldrishti-508107")

# Load the same missing-area list
csv_path = Path.home() / "Downloads" / "maharashtra_village_indices_oct2025.csv"
village_df = pd.read_csv(csv_path)

missing_ids = village_df.loc[
    village_df[["NDVI", "NDWI", "MNDWI"]].isna().all(axis=1),
    "area_id"
].dropna().tolist()

village_asset = ee.FeatureCollection(
    "projects/jaldrishti-508107/assets/maharashtra_villages"
)

missing_features = village_asset.filter(
    ee.Filter.inList("area_id", missing_ids)
)

# Count valid NDVI pixels at 20 m instead of 100 m
pixel_counts_20m = ndvi.rename("valid_pixel_count").reduceRegions(
    collection=missing_features,
    reducer=ee.Reducer.count(),
    scale=20,
    tileScale=4
)

diagnostic_20m = pixel_counts_20m.map(
    lambda feature: ee.Feature(
        None,
        feature.toDictionary([
            "area_id", "village", "district",
            "subdistric", "vlcode", "valid_pixel_count"
        ])
    )
)

task_20m = ee.batch.Export.table.toDrive(
    collection=diagnostic_20m,
    description="JalDrishti_Missing_Area_Pixel_Diagnostic_20m_Oct2025",
    folder="JalDrishti_AI",
    fileNamePrefix="missing_area_pixel_diagnostic_20m_oct2025",
    fileFormat="CSV",
    selectors=[
        "area_id", "village", "district",
        "subdistric", "vlcode", "valid_pixel_count"
    ]
)

task_20m.start()
print("20 m diagnostic started.")
print("Task ID:", task_20m.id)


20 m diagnostic started.
Task ID: SQEYPVNVO2XSE5IYH2TYM736


In [15]:
print(ee.data.getTaskStatus("SQEYPVNVO2XSE5IYH2TYM736"))

[{'state': 'COMPLETED', 'description': 'JalDrishti_Missing_Area_Pixel_Diagnostic_20m_Oct2025', 'priority': 100, 'creation_timestamp_ms': 1792258572888, 'update_timestamp_ms': 1792258589019, 'start_timestamp_ms': 1792258576923, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['https://drive.google.com/#folders/1kL9lMyylc9C8sFbOBTk6snGk0lhYtxUv'], 'attempt': 1, 'batch_eecu_usage_seconds': 656.2784423828125, 'id': 'SQEYPVNVO2XSE5IYH2TYM736', 'name': 'projects/jaldrishti-508107/operations/SQEYPVNVO2XSE5IYH2TYM736'}]


In [16]:

import ee
import pandas as pd
from pathlib import Path

ee.Initialize(project="jaldrishti-508107")

# Load the exported village results
csv_path = Path.home() / "Downloads" / "maharashtra_village_indices_oct2025.csv"
village_df = pd.read_csv(csv_path)

missing_row = village_df[
    village_df[["NDVI", "NDWI", "MNDWI"]].isna().all(axis=1)
].iloc[0]

working_row = village_df[
    village_df[["NDVI", "NDWI", "MNDWI"]].notna().all(axis=1)
].iloc[0]

village_asset = ee.FeatureCollection(
    "projects/jaldrishti-508107/assets/maharashtra_villages"
)

missing_feature = village_asset.filter(
    ee.Filter.eq("area_id", missing_row["area_id"])
)

working_feature = village_asset.filter(
    ee.Filter.eq("area_id", working_row["area_id"])
)

# Count unmasked B8 pixels before cloud masking
raw_composite = sentinel2.median()

# Count B8 pixels after cloud masking
masked_composite = sentinel2.map(mask_sentinel2_scl).median()

def get_pixel_count(image, feature_collection, band, scale=20):
    result = image.select(band).reduceRegions(
        collection=feature_collection,
        reducer=ee.Reducer.count(),
        scale=scale,
        tileScale=4
    ).first()

    return result.get("count").getInfo()

print("Missing area:", missing_row["area_id"], missing_row["village"])
print("Working area:", working_row["area_id"], working_row["village"])

print("\nMissing area — raw B8 count:",
      get_pixel_count(raw_composite, missing_feature, "B8"))
print("Missing area — masked B8 count:",
      get_pixel_count(masked_composite, missing_feature, "B8"))

print("\nWorking area — raw B8 count:",
      get_pixel_count(raw_composite, working_feature, "B8"))
print("Working area — masked B8 count:",
      get_pixel_count(masked_composite, working_feature, "B8"))


Missing area: MH_43764 Majgaon
Working area: MH_21875 Arvi

Missing area — raw B8 count: 0
Missing area — masked B8 count: 0

Working area — raw B8 count: 18309
Working area — masked B8 count: 18281


In [17]:

# Check geometry size and intersection with the October Sentinel-2 footprint

def inspect_area(feature_collection, label):
    feature = ee.Feature(feature_collection.first())
    geometry = feature.geometry()

    footprint = sentinel2.geometry()

    print("\n", label)
    print("Area (km²):", geometry.area(maxError=10).divide(1e6).getInfo())
    print("Intersects image footprint:", geometry.intersects(
        footprint, maxError=10
    ).getInfo())

inspect_area(missing_feature, "Missing area: Majgaon")
inspect_area(working_feature, "Working area: Arvi")



 Missing area: Majgaon
Area (km²): 9.715490368832054
Intersects image footprint: False

 Working area: Arvi
Area (km²): 6.932689476956527
Intersects image footprint: True


In [18]:

# Count how many missing areas intersect the October image footprint

missing_features = village_asset.filter(
    ee.Filter.inList(
        "area_id",
        missing_ids
    )
)

def add_footprint_flag(feature):
    intersects = feature.geometry().intersects(
        sentinel2.geometry(),
        maxError=10
    )
    return feature.set("intersects_footprint", intersects)

footprint_check = missing_features.map(add_footprint_flag)

footprint_summary = ee.Dictionary({
    "total_missing_areas": footprint_check.size(),
    "intersecting_footprint": footprint_check.filter(
        ee.Filter.eq("intersects_footprint", True)
    ).size(),
    "outside_footprint": footprint_check.filter(
        ee.Filter.eq("intersects_footprint", False)
    ).size()
})

print(footprint_summary.getInfo())


{'intersecting_footprint': 261, 'outside_footprint': 848, 'total_missing_areas': 1109}


In [19]:

# October 2025 Sentinel-2 collection without the scene-level cloud filter
sentinel2_all = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(maharashtra_geometry)
    .filterDate("2025-10-01", "2025-11-01")
)

print("Scenes without scene-level cloud filter:",
      sentinel2_all.size().getInfo())

# Compare collection footprint coverage for the same 1,109 missing areas
all_footprint = sentinel2_all.geometry()

def add_all_footprint_flag(feature):
    intersects = feature.geometry().intersects(
        all_footprint,
        maxError=10
    )
    return feature.set("intersects_all_footprint", intersects)

all_footprint_check = missing_features.map(add_all_footprint_flag)

print({
    "total_missing_areas": all_footprint_check.size().getInfo(),
    "intersecting_unfiltered_footprint": all_footprint_check.filter(
        ee.Filter.eq("intersects_all_footprint", True)
    ).size().getInfo(),
    "outside_unfiltered_footprint": all_footprint_check.filter(
        ee.Filter.eq("intersects_all_footprint", False)
    ).size().getInfo()
})


Scenes without scene-level cloud filter: 662
{'total_missing_areas': 1109, 'intersecting_unfiltered_footprint': 1109, 'outside_unfiltered_footprint': 0}


In [20]:
all_footprint = sentinel2_all.geometry()

def add_all_footprint_flag(feature):
    intersects = feature.geometry().intersects(
        all_footprint,
        maxError=10
    )
    return feature.set("intersects_all_footprint", intersects)

all_footprint_check = missing_features.map(add_all_footprint_flag)

print({
    "total_missing_areas": all_footprint_check.size().getInfo(),
    "intersecting_unfiltered_footprint": all_footprint_check.filter(
        ee.Filter.eq("intersects_all_footprint", True)
    ).size().getInfo(),
    "outside_unfiltered_footprint": all_footprint_check.filter(
        ee.Filter.eq("intersects_all_footprint", False)
    ).size().getInfo()
})

{'total_missing_areas': 1109, 'intersecting_unfiltered_footprint': 1109, 'outside_unfiltered_footprint': 0}


In [21]:

# Apply pixel-level SCL masking to all October scenes
# Do not apply the scene-level CLOUDY_PIXEL_PERCENTAGE filter.

sentinel2_all_masked = sentinel2_all.map(mask_sentinel2_scl)

sentinel2_all_composite = (
    sentinel2_all_masked
    .median()
    .clip(maharashtra_geometry)
)

# Recalculate indices from the broader masked composite
ndvi_all = sentinel2_all_composite.normalizedDifference(
    ["B8", "B4"]
).rename("NDVI")

ndwi_all = sentinel2_all_composite.normalizedDifference(
    ["B3", "B8"]
).rename("NDWI")

mndwi_all = sentinel2_all_composite.normalizedDifference(
    ["B3", "B11"]
).rename("MNDWI")

village_indices_all = ndvi_all.addBands(ndwi_all).addBands(mndwi_all)

print("Scenes available:", sentinel2_all.size().getInfo())
print("Masked composite bands:", village_indices_all.bandNames().getInfo())


Scenes available: 662
Masked composite bands: ['NDVI', 'NDWI', 'MNDWI']


In [22]:

# Count valid NDVI pixels using the broader, cloud-masked composite
test_counts = ndvi_all.rename("valid_pixel_count").reduceRegions(
    collection=missing_features,
    reducer=ee.Reducer.count(),
    scale=20,
    tileScale=4
)

test_counts_table = test_counts.map(
    lambda feature: ee.Feature(
        None,
        feature.toDictionary([
            "area_id", "village", "district", "valid_pixel_count"
        ])
    )
)

# Summarize the diagnostic without downloading all 1,109 rows
count_summary = ee.Dictionary({
    "total_areas": test_counts_table.size(),
    "areas_with_valid_pixels": test_counts_table.filter(
        ee.Filter.gt("valid_pixel_count", 0)
    ).size(),
    "areas_with_zero_or_missing_count": test_counts_table.filter(
        ee.Filter.Or(
            ee.Filter.eq("valid_pixel_count", 0),
            ee.Filter.notNull(["valid_pixel_count"]).Not()
        )
    ).size()
})

print(count_summary.getInfo())


{'areas_with_valid_pixels': 0, 'areas_with_zero_or_missing_count': 1109, 'total_areas': 1109}


In [23]:

# Inspect the image mask for one missing area
majgaon = village_asset.filter(
    ee.Filter.eq("area_id", "MH_43764")
).first()

print("Majgaon properties:", majgaon.toDictionary(
    ["area_id", "village", "district", "vlcode"]
).getInfo())

# Check whether the polygon intersects the image footprint
print(
    "Intersects composite footprint:",
    majgaon.geometry().intersects(
        sentinel2_all_composite.geometry(),
        maxError=10
    ).getInfo()
)

# Inspect whether each index has any unmasked pixels
for band_name in ["NDVI", "NDWI", "MNDWI"]:
    mask_image = village_indices_all.select(band_name).mask().rename("mask")
    mask_result = mask_image.reduceRegion(
        reducer=ee.Reducer.max(),
        geometry=majgaon.geometry(),
        scale=20,
        maxPixels=1_000_000,
        tileScale=4
    )
    print(band_name, mask_result.getInfo())


Majgaon properties: {'area_id': 'MH_43764', 'district': 'Sindhudurg', 'village': 'Majgaon', 'vlcode': '566922'}
Intersects composite footprint: True
NDVI {'mask': 1}
NDWI {'mask': 1}
MNDWI {'mask': 1}


In [24]:

# Diagnose one polygon with confirmed unmasked pixels
majgaon_geometry = majgaon.geometry()

print("Valid pixel counts at 20 m:")

for band_name in ["NDVI", "NDWI", "MNDWI"]:
    result = village_indices_all.select(band_name).reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=majgaon_geometry,
        scale=20,
        maxPixels=1_000_000,
        tileScale=4
    )
    print(band_name, result.getInfo())

print("\nMean index values for Majgaon:")

mean_result = village_indices_all.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=majgaon_geometry,
    scale=20,
    maxPixels=1_000_000,
    tileScale=4
)

print(mean_result.getInfo())


Valid pixel counts at 20 m:
NDVI {'NDVI': 25287}
NDWI {'NDWI': 25287}
MNDWI {'MNDWI': 25287}

Mean index values for Majgaon:
{'MNDWI': -0.4746132581701248, 'NDVI': 0.7561750876703152, 'NDWI': -0.6491708664257361}


In [25]:

# Recalculate only areas where all three original indices were missing

missing_features = village_asset.filter(
    ee.Filter.inList("area_id", missing_ids)
)

recovered_stats = village_indices_all.reduceRegions(
    collection=missing_features,
    reducer=ee.Reducer.mean(),
    scale=20,
    tileScale=4
)

# Keep only the identifiers and recalculated index values
recovered_export = recovered_stats.map(
    lambda feature: ee.Feature(
        None,
        feature.toDictionary([
            "area_id",
            "village",
            "district",
            "subdistric",
            "vlcode",
            "NDVI",
            "NDWI",
            "MNDWI"
        ])
    )
)

recovery_task = ee.batch.Export.table.toDrive(
    collection=recovered_export,
    description="JalDrishti_Recovery_Missing_Village_Indices_Oct2025",
    folder="JalDrishti_AI",
    fileNamePrefix="recovered_missing_village_indices_oct2025",
    fileFormat="CSV",
    selectors=[
        "area_id",
        "village",
        "district",
        "subdistric",
        "vlcode",
        "NDVI",
        "NDWI",
        "MNDWI"
    ]
)

recovery_task.start()

print("Recovery export started.")
print("Task ID:", recovery_task.id)
print("Areas targeted:", len(missing_ids))


Recovery export started.
Task ID: VDRDIP2KCXC3CQQRCYWHP77K
Areas targeted: 1109


In [26]:
print(ee.data.getTaskStatus("VDRDIP2KCXC3CQQRCYWHP77K"))

[{'state': 'COMPLETED', 'description': 'JalDrishti_Recovery_Missing_Village_Indices_Oct2025', 'priority': 100, 'creation_timestamp_ms': 1792259379476, 'update_timestamp_ms': 1792259404331, 'start_timestamp_ms': 1792259383591, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['https://drive.google.com/#folders/1kL9lMyylc9C8sFbOBTk6snGk0lhYtxUv'], 'attempt': 1, 'batch_eecu_usage_seconds': 1316.2005615234375, 'id': 'VDRDIP2KCXC3CQQRCYWHP77K', 'name': 'projects/jaldrishti-508107/operations/VDRDIP2KCXC3CQQRCYWHP77K'}]


In [27]:
import ee

ee.Initialize(project="jaldrishti-508107")

# CHIRPS daily precipitation for October 2025
chirps_october = (
    ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
    .filterDate("2025-10-01", "2025-11-01")
    .filterBounds(maharashtra_geometry)
)

print("CHIRPS daily images:", chirps_october.size().getInfo())

# Sum daily precipitation to get total October rainfall (mm)
rainfall_october = (
    chirps_october
    .select("precipitation")
    .sum()
    .rename("rainfall_oct2025_mm")
    .clip(maharashtra_geometry)
)

print("Rainfall band:", rainfall_october.bandNames().getInfo())

CHIRPS daily images: 31
Rainfall band: ['rainfall_oct2025_mm']


In [28]:

# Load the uploaded Maharashtra area boundaries
village_asset = ee.FeatureCollection(
    "projects/jaldrishti-508107/assets/maharashtra_villages"
)

# Calculate mean October rainfall total (mm) for each area
rainfall_stats = rainfall_october.reduceRegions(
    collection=village_asset,
    reducer=ee.Reducer.mean(),
    scale=5000,  # CHIRPS native spatial resolution is approximately 5 km
    tileScale=4
)

# Keep identifiers and the rainfall value
rainfall_export = rainfall_stats.map(
    lambda feature: ee.Feature(
        None,
        feature.toDictionary([
            "area_id",
            "village",
            "district",
            "subdistric",
            "vlcode",
            "rainfall_oct2025_mm"
        ]).set(
            "rainfall_oct2025_mm",
            feature.get("mean")
        )
    )
)

rainfall_task = ee.batch.Export.table.toDrive(
    collection=rainfall_export,
    description="JalDrishti_Maharashtra_Village_Rainfall_Oct2025",
    folder="JalDrishti_AI",
    fileNamePrefix="maharashtra_village_rainfall_oct2025",
    fileFormat="CSV",
    selectors=[
        "area_id",
        "village",
        "district",
        "subdistric",
        "vlcode",
        "rainfall_oct2025_mm"
    ]
)

rainfall_task.start()

print("Rainfall export started.")
print("Task ID:", rainfall_task.id)
print("Area records expected:", village_asset.size().getInfo())


Rainfall export started.
Task ID: 5J5GDDGUBSTZZYXUNGH55BX5
Area records expected: 44468


In [29]:

print(ee.data.getTaskStatus("5J5GDDGUBSTZZYXUNGH55BX5"))


[{'state': 'RUNNING', 'description': 'JalDrishti_Maharashtra_Village_Rainfall_Oct2025', 'priority': 100, 'creation_timestamp_ms': 1792259883124, 'update_timestamp_ms': 1792259906693, 'start_timestamp_ms': 1792259886244, 'task_type': 'EXPORT_FEATURES', 'attempt': 1, 'batch_eecu_usage_seconds': 2.107663393, 'id': '5J5GDDGUBSTZZYXUNGH55BX5', 'name': 'projects/jaldrishti-508107/operations/5J5GDDGUBSTZZYXUNGH55BX5'}]


In [30]:
print(ee.data.getTaskStatus("5J5GDDGUBSTZZYXUNGH55BX5"))

[{'state': 'FAILED', 'description': 'JalDrishti_Maharashtra_Village_Rainfall_Oct2025', 'priority': 100, 'creation_timestamp_ms': 1792259883124, 'update_timestamp_ms': 1792259954008, 'start_timestamp_ms': 1792259886244, 'task_type': 'EXPORT_FEATURES', 'attempt': 1, 'batch_eecu_usage_seconds': 125.164321899, 'error_message': "Error in map(ID=00000000000000003c46):\nDictionary.set: Parameter 'value' is required and may not be null.", 'id': '5J5GDDGUBSTZZYXUNGH55BX5', 'name': 'projects/jaldrishti-508107/operations/5J5GDDGUBSTZZYXUNGH55BX5'}]


In [31]:
# Recalculate the rainfall mean for each village/town polygon
rainfall_stats = rainfall_october.reduceRegions(
    collection=village_asset,
    reducer=ee.Reducer.mean(),
    scale=5000,
    tileScale=4
)

# Preserve the original identifiers and safely retain missing values
def prepare_rainfall_export(feature):
    return ee.Feature(
        None,
        {
            "area_id": feature.get("area_id"),
            "village": feature.get("village"),
            "district": feature.get("district"),
            "subdistric": feature.get("subdistric"),
            "vlcode": feature.get("vlcode"),
            "rainfall_oct2025_mm": feature.get("mean")
        }
    )

rainfall_export = rainfall_stats.map(prepare_rainfall_export)

rainfall_task = ee.batch.Export.table.toDrive(
    collection=rainfall_export,
    description="JalDrishti_Maharashtra_Village_Rainfall_Oct2025_v2",
    folder="JalDrishti_AI",
    fileNamePrefix="maharashtra_village_rainfall_oct2025_v2",
    fileFormat="CSV",
    selectors=[
        "area_id",
        "village",
        "district",
        "subdistric",
        "vlcode",
        "rainfall_oct2025_mm"
    ]
)

rainfall_task.start()

print("New rainfall export started.")
print("Task ID:", rainfall_task.id)

New rainfall export started.
Task ID: NVH6LC7PHNGZIJA2GRSSYVI2


In [32]:
print(ee.data.getTaskStatus(rainfall_task.id))

[{'state': 'COMPLETED', 'description': 'JalDrishti_Maharashtra_Village_Rainfall_Oct2025_v2', 'priority': 100, 'creation_timestamp_ms': 1792260009334, 'update_timestamp_ms': 1792260031374, 'start_timestamp_ms': 1792260013405, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['https://drive.google.com/#folders/1kL9lMyylc9C8sFbOBTk6snGk0lhYtxUv'], 'attempt': 1, 'batch_eecu_usage_seconds': 2.5671236515045166, 'id': 'NVH6LC7PHNGZIJA2GRSSYVI2', 'name': 'projects/jaldrishti-508107/operations/NVH6LC7PHNGZIJA2GRSSYVI2'}]


In [33]:

import ee

# Elevation from the SRTM 30 m Digital Elevation Model
dem = ee.Image("USGS/SRTMGL1_003").select("elevation")

# Derive slope in degrees
slope = ee.Terrain.slope(dem).rename("slope_deg")

# Combine the two terrain indicators
terrain_image = (
    dem.rename("elevation_m")
    .addBands(slope)
)

# Calculate mean elevation and mean slope for each polygon
terrain_stats = terrain_image.reduceRegions(
    collection=village_asset,
    reducer=ee.Reducer.mean(),
    scale=30,
    tileScale=4
)

# Prepare a clean export table
def prepare_terrain_export(feature):
    return ee.Feature(
        None,
        {
            "area_id": feature.get("area_id"),
            "village": feature.get("village"),
            "district": feature.get("district"),
            "subdistric": feature.get("subdistric"),
            "vlcode": feature.get("vlcode"),
            "elevation_mean_m": feature.get("mean"),
            "slope_mean_deg": feature.get("mean_1")
        }
    )

terrain_export = terrain_stats.map(prepare_terrain_export)

terrain_task = ee.batch.Export.table.toDrive(
    collection=terrain_export,
    description="JalDrishti_Maharashtra_Village_Terrain",
    folder="JalDrishti_AI",
    fileNamePrefix="maharashtra_village_terrain",
    fileFormat="CSV",
    selectors=[
        "area_id",
        "village",
        "district",
        "subdistric",
        "vlcode",
        "elevation_mean_m",
        "slope_mean_deg"
    ]
)

terrain_task.start()

print("Terrain export started.")
print("Task ID:", terrain_task.id)


Terrain export started.
Task ID: HGMZUMNKLHHSDHWB7DD4MQ4Z


In [34]:

import ee

# Elevation from the SRTM 30 m Digital Elevation Model
dem = ee.Image("USGS/SRTMGL1_003").select("elevation")

# Derive slope in degrees
slope = ee.Terrain.slope(dem).rename("slope_deg")

# Combine the two terrain indicators
terrain_image = (
    dem.rename("elevation_m")
    .addBands(slope)
)

# Calculate mean elevation and mean slope for each polygon
terrain_stats = terrain_image.reduceRegions(
    collection=village_asset,
    reducer=ee.Reducer.mean(),
    scale=30,
    tileScale=4
)

# Prepare a clean export table
def prepare_terrain_export(feature):
    return ee.Feature(
        None,
        {
            "area_id": feature.get("area_id"),
            "village": feature.get("village"),
            "district": feature.get("district"),
            "subdistric": feature.get("subdistric"),
            "vlcode": feature.get("vlcode"),
            "elevation_mean_m": feature.get("mean"),
            "slope_mean_deg": feature.get("mean_1")
        }
    )

terrain_export = terrain_stats.map(prepare_terrain_export)

terrain_task = ee.batch.Export.table.toDrive(
    collection=terrain_export,
    description="JalDrishti_Maharashtra_Village_Terrain",
    folder="JalDrishti_AI",
    fileNamePrefix="maharashtra_village_terrain",
    fileFormat="CSV",
    selectors=[
        "area_id",
        "village",
        "district",
        "subdistric",
        "vlcode",
        "elevation_mean_m",
        "slope_mean_deg"
    ]
)

terrain_task.start()

print("Terrain export started.")
print("Task ID:", terrain_task.id)


Terrain export started.
Task ID: M4YPC3BJXXS74XJH7BUMDRIM


In [35]:
print(ee.data.getTaskStatus("M4YPC3BJXXS74XJH7BUMDRIM"))


[{'state': 'COMPLETED', 'description': 'JalDrishti_Maharashtra_Village_Terrain', 'priority': 100, 'creation_timestamp_ms': 1792261781746, 'update_timestamp_ms': 1792261796199, 'start_timestamp_ms': 1792261784433, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['https://drive.google.com/#folders/1kL9lMyylc9C8sFbOBTk6snGk0lhYtxUv'], 'attempt': 1, 'batch_eecu_usage_seconds': 1.0926342010498047, 'id': 'M4YPC3BJXXS74XJH7BUMDRIM', 'name': 'projects/jaldrishti-508107/operations/M4YPC3BJXXS74XJH7BUMDRIM'}]


In [36]:

import ee

# ESA WorldCover 2021 at 10 m resolution
worldcover = (
    ee.ImageCollection("ESA/WorldCover/v200")
    .first()
    .select("Map")
)

# Calculate the percentage of each class within each polygon
landcover_stats = worldcover.reduceRegions(
    collection=village_asset,
    reducer=ee.Reducer.frequencyHistogram(),
    scale=10,
    tileScale=4
)

# Convert the class histogram into percentage fields
def prepare_landcover_export(feature):
    histogram = ee.Dictionary(
        ee.Algorithms.If(
            ee.Algorithms.IsEqual(feature.get("histogram"), None),
            ee.Dictionary({}),
            feature.get("histogram")
        )
    )

    total = ee.Number(
        histogram.values().reduce(ee.Reducer.sum(), [0])
    )

    classes = ee.List([
        10, 20, 30, 40, 50, 60,
        70, 80, 90, 95, 100
    ])

    class_names = ee.List([
        "tree_cover", "shrubland", "grassland",
        "cropland", "built_up", "bare_sparse",
        "snow_ice", "water", "herbaceous_wetland",
        "mangroves", "moss_lichen"
    ])

    properties = ee.Dictionary.fromLists(
        class_names,
        classes.map(
            lambda cls: ee.Algorithms.If(
                total.gt(0),
                ee.Number(
                    histogram.get(ee.Number(cls).format(), 0)
                ).divide(total).multiply(100),
                None
            )
        )
    )

    return ee.Feature(
        None,
        ee.Dictionary({
            "area_id": feature.get("area_id"),
            "village": feature.get("village"),
            "district": feature.get("district"),
            "subdistric": feature.get("subdistric"),
            "vlcode": feature.get("vlcode")
        }).combine(properties, True)
    )

landcover_export = landcover_stats.map(prepare_landcover_export)

landcover_task = ee.batch.Export.table.toDrive(
    collection=landcover_export,
    description="JalDrishti_Maharashtra_Village_LandCover_2021",
    folder="JalDrishti_AI",
    fileNamePrefix="maharashtra_village_landcover_2021",
    fileFormat="CSV"
)

landcover_task.start()

print("Land-cover export started.")
print("Task ID:", landcover_task.id)


TypeError: List.reduce() takes 2 positional arguments but 3 were given

In [37]:
histogram.values().reduce(ee.Reducer.sum(), [0])

NameError: name 'histogram' is not defined

In [38]:

def prepare_landcover_export(feature):
    histogram = ee.Dictionary(
        ee.Algorithms.If(
            ee.Algorithms.IsEqual(feature.get("histogram"), None),
            ee.Dictionary({}),
            feature.get("histogram")
        )
    )

    # Sum the pixel counts across all land-cover classes
    total = ee.Number(
        ee.List(histogram.values()).reduce(ee.Reducer.sum())
    )

    classes = ee.List([
        10, 20, 30, 40, 50, 60,
        70, 80, 90, 95, 100
    ])

    class_names = ee.List([
        "tree_cover", "shrubland", "grassland",
        "cropland", "built_up", "bare_sparse",
        "snow_ice", "water", "herbaceous_wetland",
        "mangroves", "moss_lichen"
    ])

    properties = ee.Dictionary.fromLists(
        class_names,
        classes.map(
            lambda cls: ee.Algorithms.If(
                total.gt(0),
                ee.Number(
                    histogram.get(ee.Number(cls).format(), 0)
                ).divide(total).multiply(100),
                None
            )
        )
    )

    identifiers = ee.Dictionary({
        "area_id": feature.get("area_id"),
        "village": feature.get("village"),
        "district": feature.get("district"),
        "subdistric": feature.get("subdistric"),
        "vlcode": feature.get("vlcode")
    })

    return ee.Feature(
        None,
        identifiers.combine(properties, True)
    )

landcover_export = landcover_stats.map(prepare_landcover_export)

print("Land-cover export table created successfully.")
print("Expected records:", landcover_export.size().getInfo())


Land-cover export table created successfully.


EEException: Computation timed out.

In [ ]:

landcover_task = ee.batch.Export.table.toDrive(
    collection=landcover_export,
    description="JalDrishti_Maharashtra_Village_LandCover_2021",
    folder="JalDrishti_AI",
    fileNamePrefix="maharashtra_village_landcover_2021",
    fileFormat="CSV",
    selectors=[
        "area_id",
        "village",
        "district",
        "subdistric",
        "vlcode",
        "tree_cover",
        "shrubland",
        "grassland",
        "cropland",
        "built_up",
        "bare_sparse",
        "snow_ice",
        "water",
        "herbaceous_wetland",
        "mangroves",
        "moss_lichen"
    ]
)

landcover_task.start()

print("Land-cover export started.")
print("Task ID:", landcover_task.id)



In [39]:
print(ee.data.getTaskStatus(landcover_task.id))

NameError: name 'landcover_task' is not defined

In [40]:
# Do not call landcover_export.size().getInfo()
print("Land-cover export object is ready.")

Land-cover export object is ready.


In [41]:

landcover_task = ee.batch.Export.table.toDrive(
    collection=landcover_export,
    description="JalDrishti_Maharashtra_Village_LandCover_2021",
    folder="JalDrishti_AI",
    fileNamePrefix="maharashtra_village_landcover_2021",
    fileFormat="CSV",
    selectors=[
        "area_id", "village", "district", "subdistric", "vlcode",
        "tree_cover", "shrubland", "grassland", "cropland",
        "built_up", "bare_sparse", "snow_ice", "water",
        "herbaceous_wetland", "mangroves", "moss_lichen"
    ]
)

landcover_task.start()

print("Land-cover export started.")
print("Task ID:", landcover_task.id)


Land-cover export started.
Task ID: AQXGBPE55VQDNCNUTAU7U6ZZ


In [42]:
print(ee.data.getTaskStatus("AQXGBPE55VQDNCNUTAU7U6ZZ"))

[{'state': 'RUNNING', 'description': 'JalDrishti_Maharashtra_Village_LandCover_2021', 'priority': 100, 'creation_timestamp_ms': 1792262276917, 'update_timestamp_ms': 1792262321453, 'start_timestamp_ms': 1792262281117, 'task_type': 'EXPORT_FEATURES', 'attempt': 1, 'batch_eecu_usage_seconds': 1.276175856, 'id': 'AQXGBPE55VQDNCNUTAU7U6ZZ', 'name': 'projects/jaldrishti-508107/operations/AQXGBPE55VQDNCNUTAU7U6ZZ'}]


In [43]:
print(ee.data.getTaskStatus("AQXGBPE55VQDNCNUTAU7U6ZZ"))

[{'state': 'RUNNING', 'description': 'JalDrishti_Maharashtra_Village_LandCover_2021', 'priority': 100, 'creation_timestamp_ms': 1792262276917, 'update_timestamp_ms': 1792262411443, 'start_timestamp_ms': 1792262281117, 'task_type': 'EXPORT_FEATURES', 'attempt': 1, 'batch_eecu_usage_seconds': 1.276175856, 'id': 'AQXGBPE55VQDNCNUTAU7U6ZZ', 'name': 'projects/jaldrishti-508107/operations/AQXGBPE55VQDNCNUTAU7U6ZZ'}]


In [44]:
status = ee.data.getTaskStatus("AQXGBPE55VQDNCNUTAU7U6ZZ")
print(status)

[{'state': 'RUNNING', 'description': 'JalDrishti_Maharashtra_Village_LandCover_2021', 'priority': 100, 'creation_timestamp_ms': 1792262276917, 'update_timestamp_ms': 1792262471442, 'start_timestamp_ms': 1792262281117, 'task_type': 'EXPORT_FEATURES', 'attempt': 1, 'batch_eecu_usage_seconds': 1.276175856, 'id': 'AQXGBPE55VQDNCNUTAU7U6ZZ', 'name': 'projects/jaldrishti-508107/operations/AQXGBPE55VQDNCNUTAU7U6ZZ'}]


In [45]:
print(ee.data.getTaskStatus("AQXGBPE55VQDNCNUTAU7U6ZZ"))

[{'state': 'RUNNING', 'description': 'JalDrishti_Maharashtra_Village_LandCover_2021', 'priority': 100, 'creation_timestamp_ms': 1792262276917, 'update_timestamp_ms': 1792262561444, 'start_timestamp_ms': 1792262281117, 'task_type': 'EXPORT_FEATURES', 'attempt': 1, 'batch_eecu_usage_seconds': 1.276175856, 'id': 'AQXGBPE55VQDNCNUTAU7U6ZZ', 'name': 'projects/jaldrishti-508107/operations/AQXGBPE55VQDNCNUTAU7U6ZZ'}]


In [46]:
task_id = "AQXGBPE55VQDNCNUTAU7U6ZZ"

status = ee.data.getTaskStatus(task_id)[0]

print("State:", status.get("state"))
print("Last update:", status.get("update_timestamp_ms"))
print("Compute time:", status.get("batch_eecu_usage_seconds"))
print("Error:", status.get("error_message", "No error reported"))

State: RUNNING
Last update: 1792262581443
Compute time: 1.276175856
Error: No error reported


In [47]:
print(ee.data.getTaskStatus("AQXGBPE55VQDNCNUTAU7U6ZZ"))

[{'state': 'RUNNING', 'description': 'JalDrishti_Maharashtra_Village_LandCover_2021', 'priority': 100, 'creation_timestamp_ms': 1792262276917, 'update_timestamp_ms': 1792262611442, 'start_timestamp_ms': 1792262281117, 'task_type': 'EXPORT_FEATURES', 'attempt': 1, 'batch_eecu_usage_seconds': 1.276175856, 'id': 'AQXGBPE55VQDNCNUTAU7U6ZZ', 'name': 'projects/jaldrishti-508107/operations/AQXGBPE55VQDNCNUTAU7U6ZZ'}]


In [48]:
[{'state': 'RUNNING', 'description': 'JalDrishti_Maharashtra_Village_LandCover_2021', 'priority': 100, 'creation_timestamp_ms': 1792262276917, 'update_timestamp_ms': 1792262611442, 'start_timestamp_ms': 1792262281117, 'task_type': 'EXPORT_FEATURES', 'attempt': 1, 'batch_eecu_usage_seconds': 1.276175856, 'id': 'AQXGBPE55VQDNCNUTAU7U6ZZ', 'name': 'projects/jaldrishti-508107/operations/AQXGBPE55VQDNCNUTAU7U6ZZ'}]

[{'state': 'RUNNING',
  'description': 'JalDrishti_Maharashtra_Village_LandCover_2021',
  'priority': 100,
  'creation_timestamp_ms': 1792262276917,
  'update_timestamp_ms': 1792262611442,
  'start_timestamp_ms': 1792262281117,
  'task_type': 'EXPORT_FEATURES',
  'attempt': 1,
  'batch_eecu_usage_seconds': 1.276175856,
  'id': 'AQXGBPE55VQDNCNUTAU7U6ZZ',
  'name': 'projects/jaldrishti-508107/operations/AQXGBPE55VQDNCNUTAU7U6ZZ'}]

In [49]:
print("New batch task:")
print(landcover_batch_task.id)

print("\nBatch status:")
print(ee.data.getTaskStatus(landcover_batch_task.id))

New batch task:


NameError: name 'landcover_batch_task' is not defined

In [50]:
print(ee.data.getTaskStatus("AQXGBPE55VQDNCNUTAU7U6ZZ"))

[{'state': 'RUNNING', 'description': 'JalDrishti_Maharashtra_Village_LandCover_2021', 'priority': 100, 'creation_timestamp_ms': 1792262276917, 'update_timestamp_ms': 1792262841444, 'start_timestamp_ms': 1792262281117, 'task_type': 'EXPORT_FEATURES', 'attempt': 1, 'batch_eecu_usage_seconds': 686.04901123, 'id': 'AQXGBPE55VQDNCNUTAU7U6ZZ', 'name': 'projects/jaldrishti-508107/operations/AQXGBPE55VQDNCNUTAU7U6ZZ'}]


In [51]:
print(ee.data.getTaskStatus("AQXGBPE55VQDNCNUTAU7U6ZZ"))

[{'state': 'RUNNING', 'description': 'JalDrishti_Maharashtra_Village_LandCover_2021', 'priority': 100, 'creation_timestamp_ms': 1792262276917, 'update_timestamp_ms': 1792262951446, 'start_timestamp_ms': 1792262281117, 'task_type': 'EXPORT_FEATURES', 'attempt': 1, 'batch_eecu_usage_seconds': 686.04901123, 'id': 'AQXGBPE55VQDNCNUTAU7U6ZZ', 'name': 'projects/jaldrishti-508107/operations/AQXGBPE55VQDNCNUTAU7U6ZZ'}]


In [52]:
tasks = ee.data.getTaskList()

for task in tasks:
    if task.get("id") == "AQXGBPE55VQDNCNUTAU7U6ZZ":
        print(task)

{'state': 'COMPLETED', 'description': 'JalDrishti_Maharashtra_Village_LandCover_2021', 'priority': 100, 'creation_timestamp_ms': 1792262276917, 'update_timestamp_ms': 1792263105814, 'start_timestamp_ms': 1792262281117, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['https://drive.google.com/#folders/1kL9lMyylc9C8sFbOBTk6snGk0lhYtxUv'], 'attempt': 1, 'batch_eecu_usage_seconds': 54588.62890625, 'id': 'AQXGBPE55VQDNCNUTAU7U6ZZ', 'name': 'projects/jaldrishti-508107/operations/AQXGBPE55VQDNCNUTAU7U6ZZ'}


In [53]:

import ee

# Check whether the SoilGrids image collection is available
try:
    soilgrids = ee.Image(
        "projects/soilgrids-isric/sand_mean"
    )

    print("SoilGrids image loaded.")
    print("Available bands:", soilgrids.bandNames().getInfo())

except Exception as e:
    print("SoilGrids check failed:")
    print(e)


SoilGrids image loaded.
Available bands: ['sand_0-5cm_mean', 'sand_5-15cm_mean', 'sand_15-30cm_mean', 'sand_30-60cm_mean', 'sand_60-100cm_mean', 'sand_100-200cm_mean']


In [54]:

soil_assets = {
    "sand": "projects/soilgrids-isric/sand_mean",
    "clay": "projects/soilgrids-isric/clay_mean",
    "silt": "projects/soilgrids-isric/silt_mean",
    "soil_organic_carbon": "projects/soilgrids-isric/soc_mean",
}

for name, asset_id in soil_assets.items():
    try:
        image = ee.Image(asset_id)
        bands = image.bandNames().getInfo()
        print(f"\n{name}: AVAILABLE")
        print("Bands:", bands)
    except Exception as e:
        print(f"\n{name}: NOT AVAILABLE")
        print(str(e)[:300])



sand: AVAILABLE
Bands: ['sand_0-5cm_mean', 'sand_5-15cm_mean', 'sand_15-30cm_mean', 'sand_30-60cm_mean', 'sand_60-100cm_mean', 'sand_100-200cm_mean']

clay: AVAILABLE
Bands: ['clay_0-5cm_mean', 'clay_5-15cm_mean', 'clay_15-30cm_mean', 'clay_30-60cm_mean', 'clay_60-100cm_mean', 'clay_100-200cm_mean']

silt: AVAILABLE
Bands: ['silt_0-5cm_mean', 'silt_5-15cm_mean', 'silt_15-30cm_mean', 'silt_30-60cm_mean', 'silt_60-100cm_mean', 'silt_100-200cm_mean']

soil_organic_carbon: AVAILABLE
Bands: ['soc_0-5cm_mean', 'soc_5-15cm_mean', 'soc_15-30cm_mean', 'soc_30-60cm_mean', 'soc_60-100cm_mean', 'soc_100-200cm_mean']


In [55]:

import ee

# Load SoilGrids images
sand = ee.Image("projects/soilgrids-isric/sand_mean")
clay = ee.Image("projects/soilgrids-isric/clay_mean")
silt = ee.Image("projects/soilgrids-isric/silt_mean")
soc = ee.Image("projects/soilgrids-isric/soc_mean")

# Select topsoil and shallow subsoil layers
soil_image = (
    sand.select(
        ["sand_0-5cm_mean", "sand_5-15cm_mean"],
        ["sand_0_5cm", "sand_5_15cm"]
    )
    .addBands(
        clay.select(
            ["clay_0-5cm_mean", "clay_5-15cm_mean"],
            ["clay_0_5cm", "clay_5_15cm"]
        )
    )
    .addBands(
        silt.select(
            ["silt_0-5cm_mean", "silt_5-15cm_mean"],
            ["silt_0_5cm", "silt_5_15cm"]
        )
    )
    .addBands(
        soc.select(
            ["soc_0-5cm_mean", "soc_5-15cm_mean"],
            ["soc_0_5cm", "soc_5_15cm"]
        )
    )
)

# Calculate mean soil properties for each geographic area
soil_stats = soil_image.reduceRegions(
    collection=village_asset,
    reducer=ee.Reducer.mean(),
    scale=250,
    tileScale=4
)

print("Soil statistics table prepared.")


Soil statistics table prepared.


In [56]:

soil_task = ee.batch.Export.table.toDrive(
    collection=soil_stats,
    description="JalDrishti_Maharashtra_Village_SoilGrids",
    folder="JalDrishti_AI",
    fileNamePrefix="maharashtra_village_soilgrids",
    fileFormat="CSV",
    selectors=[
        "area_id", "village", "district", "subdistric", "vlcode",
        "sand_0_5cm", "sand_5_15cm",
        "clay_0_5cm", "clay_5_15cm",
        "silt_0_5cm", "silt_5_15cm",
        "soc_0_5cm", "soc_5_15cm"
    ]
)

soil_task.start()

print("Soil export started.")
print("Task ID:", soil_task.id)


Soil export started.
Task ID: WT5KWPMM7P57LQI25CBAZMHA


In [57]:
print(ee.data.getTaskStatus(soil_task.id))

[{'state': 'COMPLETED', 'description': 'JalDrishti_Maharashtra_Village_SoilGrids', 'priority': 100, 'creation_timestamp_ms': 1792299605678, 'update_timestamp_ms': 1792299611428, 'start_timestamp_ms': 1792299608129, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['https://drive.google.com/#folders/1kL9lMyylc9C8sFbOBTk6snGk0lhYtxUv'], 'attempt': 1, 'batch_eecu_usage_seconds': 1.6976368427276611, 'id': 'WT5KWPMM7P57LQI25CBAZMHA', 'name': 'projects/jaldrishti-508107/operations/WT5KWPMM7P57LQI25CBAZMHA'}]


In [58]:

import ee

# CHIRPS daily rainfall, in millimetres per day
chirps_daily = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")

# Build annual accumulated rainfall images for 2001–2025
annual_images = []

for year in range(2001, 2026):
    annual_rain = (
        chirps_daily
        .filterDate(f"{year}-01-01", f"{year + 1}-01-01")
        .select("precipitation")
        .sum()
        .rename(f"rainfall_{year}_mm")
    )
    annual_images.append(annual_rain)

# Combine annual totals into a multi-band image
annual_rainfall = ee.Image.cat(annual_images)

# Long-term baseline: mean annual rainfall, 2001–2020
baseline_images = ee.ImageCollection.fromImages(
    annual_images[:20]
)

baseline_mean = baseline_images.mean().rename(
    "baseline_mean_annual_mm_2001_2020"
)

baseline_sd = baseline_images.reduce(
    ee.Reducer.stdDev()
).rename("baseline_sd_annual_mm_2001_2020")

# 2025 rainfall anomaly relative to the 2001–2020 baseline
rainfall_2025 = annual_images[24].rename("rainfall_2025_mm")

anomaly_2025 = rainfall_2025.subtract(
    baseline_mean
).rename("rainfall_anomaly_2025_mm")

anomaly_pct_2025 = (
    anomaly_2025.divide(baseline_mean)
    .multiply(100)
    .updateMask(baseline_mean.neq(0))
    .rename("rainfall_anomaly_2025_pct")
)

print("Annual rainfall images prepared: 2001–2025")
print("Baseline period: 2001–2020")
print("Anomaly year: 2025")


Annual rainfall images prepared: 2001–2025
Baseline period: 2001–2020
Anomaly year: 2025


In [59]:

historical_rainfall_image = (
    annual_rainfall
    .addBands(baseline_mean)
    .addBands(baseline_sd)
    .addBands(anomaly_2025)
    .addBands(anomaly_pct_2025)
)

historical_rainfall_stats = historical_rainfall_image.reduceRegions(
    collection=village_asset,
    reducer=ee.Reducer.mean(),
    scale=5500,
    tileScale=4
)

historical_rainfall_task = ee.batch.Export.table.toDrive(
    collection=historical_rainfall_stats,
    description="JalDrishti_Maharashtra_Historical_Rainfall",
    folder="JalDrishti_AI",
    fileNamePrefix="maharashtra_village_historical_rainfall_2001_2025",
    fileFormat="CSV"
)

historical_rainfall_task.start()

print("Historical rainfall export started.")
print("Task ID:", historical_rainfall_task.id)


Historical rainfall export started.
Task ID: GDFCJR7DGVDGXH2WEUSYGEJL


In [60]:
print(ee.data.getTaskStatus(historical_rainfall_task.id))

[{'state': 'RUNNING', 'description': 'JalDrishti_Maharashtra_Historical_Rainfall', 'priority': 100, 'creation_timestamp_ms': 1792299708306, 'update_timestamp_ms': 1792299710718, 'start_timestamp_ms': 1792299710509, 'task_type': 'EXPORT_FEATURES', 'attempt': 1, 'id': 'GDFCJR7DGVDGXH2WEUSYGEJL', 'name': 'projects/jaldrishti-508107/operations/GDFCJR7DGVDGXH2WEUSYGEJL'}]


In [61]:
print(ee.data.getTaskStatus("GDFCJR7DGVDGXH2WEUSYGEJL"))

[{'state': 'FAILED', 'description': 'JalDrishti_Maharashtra_Historical_Rainfall', 'priority': 100, 'creation_timestamp_ms': 1792299708306, 'update_timestamp_ms': 1792299727178, 'start_timestamp_ms': 1792299710509, 'task_type': 'EXPORT_FEATURES', 'attempt': 1, 'batch_eecu_usage_seconds': 22.09377861, 'error_message': 'Expected a homogeneous image collection, but an image with incompatible bands was encountered:\n  First image type: 1 bands ([rainfall_2001_mm]).\nCurrent image type: 1 bands ([rainfall_2002_mm]).\n          Image ID: 1\nSome bands might require explicit casts.', 'id': 'GDFCJR7DGVDGXH2WEUSYGEJL', 'name': 'projects/jaldrishti-508107/operations/GDFCJR7DGVDGXH2WEUSYGEJL'}]


In [62]:

import ee

chirps_daily = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")

# Build annual rainfall totals using a consistent band name
annual_images = []

for year in range(2001, 2026):
    annual_rain = (
        chirps_daily
        .filterDate(f"{year}-01-01", f"{year + 1}-01-01")
        .select("precipitation")
        .sum()
        .rename("annual_rainfall_mm")
        .set("year", year)
    )
    annual_images.append(annual_rain)

annual_collection = ee.ImageCollection.fromImages(annual_images)

# Calculate 2001–2020 baseline
baseline_collection = annual_collection.filter(
    ee.Filter.rangeContains("year", 2001, 2020)
)

baseline_mean = baseline_collection.mean().rename(
    "baseline_mean_annual_mm_2001_2020"
)

baseline_sd = baseline_collection.reduce(
    ee.Reducer.stdDev()
).rename("baseline_sd_annual_mm_2001_2020")

# Select the 2025 annual total
rainfall_2025 = ee.Image(
    annual_collection.filter(ee.Filter.eq("year", 2025)).first()
).rename("rainfall_2025_mm")

# Calculate absolute and percentage anomalies
anomaly_2025 = rainfall_2025.subtract(
    baseline_mean
).rename("rainfall_anomaly_2025_mm")

anomaly_pct_2025 = (
    anomaly_2025.divide(baseline_mean)
    .multiply(100)
    .updateMask(baseline_mean.neq(0))
    .rename("rainfall_anomaly_2025_pct")
)

# Restore distinct names for the annual-total bands
annual_bands = [
    ee.Image(
        annual_collection.filter(ee.Filter.eq("year", year)).first()
    ).rename(f"rainfall_{year}_mm")
    for year in range(2001, 2026)
]

annual_rainfall = ee.Image.cat(annual_bands)

historical_rainfall_image = (
    annual_rainfall
    .addBands(baseline_mean)
    .addBands(baseline_sd)
    .addBands(anomaly_2025)
    .addBands(anomaly_pct_2025)
)

print("Historical rainfall image prepared.")
print("Years: 2001–2025")
print("Baseline: 2001–2020")



Historical rainfall image prepared.
Years: 2001–2025
Baseline: 2001–2020


In [63]:

historical_rainfall_stats = historical_rainfall_image.reduceRegions(
    collection=village_asset,
    reducer=ee.Reducer.mean(),
    scale=5500,
    tileScale=4
)

historical_rainfall_task = ee.batch.Export.table.toDrive(
    collection=historical_rainfall_stats,
    description="JalDrishti_Maharashtra_Historical_Rainfall_v2",
    folder="JalDrishti_AI",
    fileNamePrefix="maharashtra_village_historical_rainfall_2001_2025_v2",
    fileFormat="CSV"
)

historical_rainfall_task.start()

print("Corrected historical rainfall export started.")
print("Task ID:", historical_rainfall_task.id)


Corrected historical rainfall export started.
Task ID: 4MYWOOQ3YFDZJAV24E4RQ3GG


In [64]:
print(ee.data.getTaskStatus(historical_rainfall_task.id))

[{'state': 'RUNNING', 'description': 'JalDrishti_Maharashtra_Historical_Rainfall_v2', 'priority': 100, 'creation_timestamp_ms': 1792299881249, 'update_timestamp_ms': 1792299885388, 'start_timestamp_ms': 1792299884947, 'task_type': 'EXPORT_FEATURES', 'attempt': 1, 'id': '4MYWOOQ3YFDZJAV24E4RQ3GG', 'name': 'projects/jaldrishti-508107/operations/4MYWOOQ3YFDZJAV24E4RQ3GG'}]


In [65]:
print(ee.data.getTaskStatus("4MYWOOQ3YFDZJAV24E4RQ3GG"))

[{'state': 'RUNNING', 'description': 'JalDrishti_Maharashtra_Historical_Rainfall_v2', 'priority': 100, 'creation_timestamp_ms': 1792299881249, 'update_timestamp_ms': 1792299915400, 'start_timestamp_ms': 1792299884947, 'task_type': 'EXPORT_FEATURES', 'attempt': 1, 'batch_eecu_usage_seconds': 22.066331863, 'id': '4MYWOOQ3YFDZJAV24E4RQ3GG', 'name': 'projects/jaldrishti-508107/operations/4MYWOOQ3YFDZJAV24E4RQ3GG'}]


In [66]:
print(ee.data.getTaskStatus("4MYWOOQ3YFDZJAV24E4RQ3GG"))

[{'state': 'RUNNING', 'description': 'JalDrishti_Maharashtra_Historical_Rainfall_v2', 'priority': 100, 'creation_timestamp_ms': 1792299881249, 'update_timestamp_ms': 1792299945397, 'start_timestamp_ms': 1792299884947, 'task_type': 'EXPORT_FEATURES', 'attempt': 1, 'batch_eecu_usage_seconds': 224.869766235, 'id': '4MYWOOQ3YFDZJAV24E4RQ3GG', 'name': 'projects/jaldrishti-508107/operations/4MYWOOQ3YFDZJAV24E4RQ3GG'}]


In [67]:
print(ee.data.getTaskStatus("4MYWOOQ3YFDZJAV24E4RQ3GG"))

[{'state': 'COMPLETED', 'description': 'JalDrishti_Maharashtra_Historical_Rainfall_v2', 'priority': 100, 'creation_timestamp_ms': 1792299881249, 'update_timestamp_ms': 1792299979755, 'start_timestamp_ms': 1792299884947, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['https://drive.google.com/#folders/1kL9lMyylc9C8sFbOBTk6snGk0lhYtxUv'], 'attempt': 1, 'batch_eecu_usage_seconds': 226.8463134765625, 'id': '4MYWOOQ3YFDZJAV24E4RQ3GG', 'name': 'projects/jaldrishti-508107/operations/4MYWOOQ3YFDZJAV24E4RQ3GG'}]
